# 07 — Parameter Tuning

Interactive exploration of key hyperparameters and their impact on forecasts.

Parameters explored:
- **Decay rate** — how much recent transitions are weighted
- **Adaptive threshold multiplier** — 0.5 × median(|returns|) controls regime sensitivity
- **Horizon** — forecast days ahead
- **Dirichlet alpha** — smoothing prior for transition counts
- **Structural break threshold** — Frobenius divergence cutoff

**Prerequisites:**
```bash
conda activate cramer-research
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, fixed

from research.data.prices import fetch_historical_prices
from research.models.markov import (
    classify_regime_series,
    estimate_transition_matrix,
    compute_markov_forecast,
    detect_structural_break,
)

## 1. Load Data

Fetch a fixed dataset to use across all tuning experiments.

In [ ]:
TICKER = 'BTC'
DAYS = 180

prices = fetch_historical_prices(TICKER, days=DAYS)
prices['returns'] = prices['close'].pct_change()
returns = prices['returns'].dropna().values
current_price = float(prices['close'].iloc[-1])

print(f"Loaded {len(returns)} returns for {TICKER}, current price ${current_price:,.2f}")

## 2. Interactive Decay Rate Slider

Explore how the exponential decay rate affects transition matrices and forecasts.

In [ ]:
def _up_rates(regimes, returns, horizon, decay_rate=0.97):
    counts = {s: {'up': 0.0, 'total': 0.0} for s in ['bull', 'bear', 'sideways']}
    max_start = min(len(regimes), len(returns)) - horizon
    for i in range(max_start):
        regime = regimes[i]
        cum_return = sum(returns[i+1:i+1+horizon])
        weight = decay_rate ** (max_start - 1 - i)
        counts[regime]['total'] += weight
        if cum_return > 0:
            counts[regime]['up'] += weight
    return {
        s: counts[s]['up'] / counts[s]['total'] if counts[s]['total'] > 0 else 0.5
        for s in ['bull', 'bear', 'sideways']
    }


@interact(
    decay=FloatSlider(min=0.90, max=0.99, step=0.01, value=0.97, description='Decay:'),
    horizon=IntSlider(min=1, max=30, step=1, value=7, description='Horizon:'),
)
def explore_decay(decay, horizon):
    regimes = classify_regime_series(returns)
    P = estimate_transition_matrix(regimes, decay_rate=decay)
    forecast = compute_markov_forecast(P, regimes[-1], horizon)
    up = _up_rates(regimes, returns, horizon, decay)
    p_up = sum(forecast[s] * up[s] for s in ['bull', 'bear', 'sideways'])
    
    # Plot transition matrix
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    states = ['bull', 'bear', 'sideways']
    im = axes[0].imshow(P, cmap='Blues', vmin=0, vmax=1)
    axes[0].set_xticks(range(3))
    axes[0].set_yticks(range(3))
    axes[0].set_xticklabels(states)
    axes[0].set_yticklabels(states)
    axes[0].set_xlabel('To')
    axes[0].set_ylabel('From')
    axes[0].set_title(f'Transition Matrix (decay={decay})')
    for i in range(3):
        for j in range(3):
            axes[0].text(j, i, f'{P[i,j]:.3f}', ha='center', va='center')
    plt.colorbar(im, ax=axes[0])
    
    # Plot regime probabilities at horizon
    bars = axes[1].bar(states, [forecast[s] for s in states], color=['green', 'red', 'gray'], alpha=0.7)
    axes[1].axhline(1/3, color='black', linestyle='--', alpha=0.3, label='Uniform')
    axes[1].set_ylabel('Probability')
    axes[1].set_title(f'Regime Probabilities at Horizon={horizon}')
    axes[1].set_ylim(0, 1)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # Add text annotations
    for bar, state in zip(bars, states):
        height = bar.get_height()
        axes[1].text(bar.get_x() + bar.get_width()/2., height + 0.02,
                    f'{height:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    print(f"P(up) = {p_up:.4f}  ({'BULLISH' if p_up > 0.5 else 'BEARISH'})")

## 3. Grid Search: Decay × Threshold

Systematic sweep across decay rate and adaptive threshold multiplier.

In [ ]:
decay_values = np.linspace(0.90, 0.99, 10)
threshold_multipliers = np.linspace(0.3, 0.8, 6)
HORIZON = 7

grid_results = []
for decay in decay_values:
    for mult in threshold_multipliers:
        regimes = classify_regime_series(returns, return_threshold_multiplier=mult)
        P = estimate_transition_matrix(regimes, decay_rate=decay)
        forecast = compute_markov_forecast(P, regimes[-1], HORIZON)
        up = _up_rates(regimes, returns, HORIZON, decay)
        p_up = sum(forecast[s] * up[s] for s in ['bull', 'bear', 'sideways'])
        
        grid_results.append({
            'decay': decay,
            'threshold_mult': mult,
            'p_up': p_up,
            'bull_prob': forecast['bull'],
            'bear_prob': forecast['bear'],
            'sideways_prob': forecast['sideways'],
        })

grid_df = pd.DataFrame(grid_results)

# Heatmap of P(up)
pivot = grid_df.pivot(index='threshold_mult', columns='decay', values='p_up')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0.3, vmax=0.7,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'P(up)'})
ax.set_title(f'{TICKER}: P(up) Grid Search (Decay × Threshold Multiplier, H={HORIZON})')
ax.set_xlabel('Decay Rate')
ax.set_ylabel('Threshold Multiplier')
plt.tight_layout()
plt.show()

## 4. Structural Break Sensitivity

How sensitive is break detection to the divergence threshold?

In [ ]:
thresholds = np.linspace(0.01, 0.15, 20)
regimes = classify_regime_series(returns)

break_results = []
for thresh in thresholds:
    result = detect_structural_break(regimes, divergence_threshold=thresh, decay_rate=0.97)
    break_results.append({
        'threshold': thresh,
        'detected': result['detected'],
        'divergence': result['divergence'],
    })

break_df = pd.DataFrame(break_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(break_df['threshold'], break_df['divergence'], linewidth=2, label='Divergence')
ax.axhline(break_df['divergence'].mean(), color='gray', linestyle='--', alpha=0.5, label=f'Mean={break_df['divergence'].mean():.4f}')
ax.axvline(0.05, color='red', linestyle=':', alpha=0.7, label='Default threshold (0.05)')
ax.set_xlabel('Divergence Threshold')
ax.set_ylabel('Frobenius Divergence')
ax.set_title(f'{TICKER}: Structural Break Sensitivity')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show where break is detected
detected_at = break_df[break_df['detected']]['threshold'].values
if len(detected_at) > 0:
    print(f"Break detected for thresholds: [{detected_at[0]:.3f}, {detected_at[-1]:.3f}]")
    print(f"Break NOT detected for thresholds > {detected_at[-1]:.3f}")
else:
    print("No break detected at any threshold")

## 5. Horizon Sensitivity with Multiple Decay Rates

Compare forecasts across horizons for different decay rates.

In [ ]:
horizons = list(range(1, 31))
decay_rates = [0.92, 0.95, 0.97, 0.99]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, decay in enumerate(decay_rates):
    ax = axes[idx]
    p_up_values = []
    
    for h in horizons:
        regimes = classify_regime_series(returns)
        P = estimate_transition_matrix(regimes, decay_rate=decay)
        forecast = compute_markov_forecast(P, regimes[-1], h)
        up = _up_rates(regimes, returns, h, decay)
        p_up = sum(forecast[s] * up[s] for s in ['bull', 'bear', 'sideways'])
        p_up_values.append(p_up)
    
    ax.plot(horizons, p_up_values, linewidth=2, label=f'decay={decay}')
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Neutral')
    ax.fill_between(horizons, 0.5, p_up_values, where=[p >= 0.5 for p in p_up_values],
                     alpha=0.2, color='green', label='Bullish zone')
    ax.fill_between(horizons, 0.5, p_up_values, where=[p < 0.5 for p in p_up_values],
                     alpha=0.2, color='red', label='Bearish zone')
    ax.set_xlabel('Horizon (days)')
    ax.set_ylabel('P(up)')
    ax.set_title(f'Decay Rate = {decay}')
    ax.set_ylim(0.3, 0.7)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'{TICKER}: P(up) Across Horizons for Different Decay Rates', y=1.01)
plt.tight_layout()
plt.show()

## 6. Walk-Forward Calibration

Test which decay rate maximised directional accuracy in a walk-forward backtest.

In [ ]:
def walk_forward_accuracy(returns, decay, horizon, threshold_mult=0.5, warmup=60, stride=5):
    """Walk-forward directional accuracy."""
    correct = 0
    total = 0
    for i in range(warmup, len(returns) - horizon, stride):
        window = returns[i-warmup:i]
        window_regimes = classify_regime_series(window, return_threshold_multiplier=threshold_mult)
        P = estimate_transition_matrix(window_regimes, decay_rate=decay)
        forecast = compute_markov_forecast(P, window_regimes[-1], horizon)
        up = _up_rates(window_regimes, window, horizon, decay)
        p_up = sum(forecast[s] * up[s] for s in ['bull', 'bear', 'sideways'])
        
        pred = 1 if p_up > 0.5 else -1
        actual = 1 if sum(returns[i:i+horizon]) > 0 else -1
        if pred == actual:
            correct += 1
        total += 1
    return correct / total if total > 0 else 0.5

# Test decay rates
calibration_decays = np.linspace(0.90, 0.99, 10)
HORIZON = 7

calib_results = []
for decay in calibration_decays:
    acc = walk_forward_accuracy(returns, decay, HORIZON)
    calib_results.append({'decay': decay, 'accuracy': acc})

calib_df = pd.DataFrame(calib_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(calib_df['decay'], calib_df['accuracy'] * 100, marker='o', linewidth=2)
ax.axhline(50, color='gray', linestyle='--', alpha=0.5, label='Random (50%)')
best = calib_df.loc[calib_df['accuracy'].idxmax()]
ax.axvline(best['decay'], color='red', linestyle=':', alpha=0.7, label=f'Best: decay={best['decay']:.3f} ({best['accuracy']*100:.1f}%)')
ax.set_xlabel('Decay Rate')
ax.set_ylabel('Directional Accuracy (%)')
ax.set_title(f'{TICKER}: Walk-Forward Calibration (H={HORIZON})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(calib_df.to_string(index=False))

## 7. Recommended Parameters

Summary of parameter recommendations from backtesting and literature.

In [ ]:
recommendations = pd.DataFrame([
    {
        'Parameter': 'Decay Rate',
        'Default': '0.97',
        'Range': '[0.90, 0.99]',
        'Recommended': f'{best['decay']:.3f}' if 'best' in locals() else '0.97',
        'Rationale': 'Higher = more weight to recent transitions; 0.97 balances stability and responsiveness',
    },
    {
        'Parameter': 'Adaptive Threshold Multiplier',
        'Default': '0.5',
        'Range': '[0.3, 0.8]',
        'Recommended': '0.5',
        'Rationale': 'Half-median of absolute returns ensures ~30-40% bull/bear each',
    },
    {
        'Parameter': 'Dirichlet Alpha',
        'Default': 'auto: max(0.01, 5/N)',
        'Range': '[0.01, 0.10]',
        'Recommended': 'auto',
        'Rationale': 'Auto-tuned scales inversely with sample size; prevents overfitting',
    },
    {
        'Parameter': 'Structural Break Threshold',
        'Default': '0.05',
        'Range': '[0.01, 0.15]',
        'Recommended': '0.05',
        'Rationale': 'Frobenius divergence; 0.05 catches meaningful regime shifts without false positives',
    },
    {
        'Parameter': 'Horizon',
        'Default': '7',
        'Range': '[1, 30]',
        'Recommended': '7-14',
        'Rationale': 'Short horizons more accurate; longer horizons widen CI significantly',
    },
    {
        'Parameter': 'Warmup Window',
        'Default': '60',
        'Range': '[30, 120]',
        'Recommended': '60-90',
        'Rationale': 'Need enough data for stable transition estimates; 60 ≈ 3 months',
    },
])

print(recommendations.to_string(index=False))